# Show2D — Pipeline Profiling
Instruments each stage: Python init, comm transfer, JS processing.
Run cells top-to-bottom and read the timing output.

In [ ]:
try:
    %load_ext autoreload
    %autoreload 2
    %env ANYWIDGET_HMR=1
except Exception:
    pass

In [ ]:
import numpy as np
import torch
import time
import sys
from quantem.widget import Show2D, profile
from quantem.widget.show2d import Show2D as _Show2D
from quantem.widget.array_utils import to_numpy

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

def make_em_image(h, w, seed=0):
    rng = np.random.default_rng(seed)
    y, x = torch.meshgrid(
        torch.arange(h, device=device, dtype=torch.float32),
        torch.arange(w, device=device, dtype=torch.float32), indexing="ij")
    img = (1.0 - y / h) * 2.0 + torch.cos(2 * np.pi * 0.03 * (x * np.cos(seed) + y * np.sin(seed)))
    noise = torch.from_numpy(rng.normal(0, 0.15, (h, w)).astype(np.float32)).to(device)
    return (img + noise).cpu().numpy()

profile()

## Stage-by-stage profiling: 10 × 4096×4096
Break down where time is spent in Python before the widget even reaches JS.

In [ ]:
# Stage 1: Generate images
t0 = time.perf_counter()
imgs = [make_em_image(4096, 4096, seed=i) for i in range(10)]
t1 = time.perf_counter()
print(f"1. Generate 10 images:  {t1-t0:.3f}s")

# Stage 2: to_numpy + stack
t2 = time.perf_counter()
arrays = [to_numpy(img) for img in imgs]
stacked = np.stack(arrays)
t3 = time.perf_counter()
print(f"2. Stack to ndarray:    {t3-t2:.3f}s  | shape={stacked.shape}")

# Stage 3: astype float32
t4 = time.perf_counter()
data = stacked.astype(np.float32)
t5 = time.perf_counter()
print(f"3. astype(float32):     {t5-t4:.3f}s  | {data.nbytes/1e6:.0f} MB")

# Stage 4: _data_original copy
t6 = time.perf_counter()
copies = [data[i].copy() for i in range(data.shape[0])]
t7 = time.perf_counter()
print(f"4. _data_original copy: {t7-t6:.3f}s  | {sum(c.nbytes for c in copies)/1e6:.0f} MB extra")

# Stage 5: Compute stats (mean/min/max/std for all 10)
t8 = time.perf_counter()
for i in range(10):
    _ = float(np.mean(data[i]))
    _ = float(np.min(data[i]))
    _ = float(np.max(data[i]))
    _ = float(np.std(data[i]))
t9 = time.perf_counter()
print(f"5. Compute all stats:   {t9-t8:.3f}s")

# Stage 6: .tobytes()
t10 = time.perf_counter()
blob = data.tobytes()
t11 = time.perf_counter()
print(f"6. .tobytes():          {t11-t10:.3f}s  | {len(blob)/1e6:.0f} MB blob")

# Total Python-side
print(f"\n   Total Python:        {t11-t2:.3f}s")
print(f"   Total with gen:      {t11-t0:.3f}s")
print(f"\n   After this, Jupyter sends {len(blob)/1e6:.0f} MB through the websocket.")

## Full widget creation (with JS render)
Time the full `Show2D(imgs)` call vs just displaying it.
The gap between init and display is the comm transfer + JS processing.

In [ ]:
# Reuse imgs from above
t0 = time.perf_counter()
w = Show2D(imgs, title="10× 4096×4096", cmap="inferno")
t1 = time.perf_counter()
print(f"Python init:     {t1-t0:.3f}s")
print(f"frame_bytes:     {len(w.frame_bytes)/1e6:.0f} MB")
print()
print("Displaying widget now — watch for delay before it renders...")
print("That delay = Jupyter comm transfer + JS decode + colormap + canvas render")
w

## Comparison: 1 image vs 10 images at 4096×4096
Isolates whether the bottleneck scales with data size.

In [ ]:
# Single 4K image
t0 = time.perf_counter()
w_single = Show2D(imgs[0], title="1× 4096×4096", cmap="inferno")
t1 = time.perf_counter()
print(f"1 image  — Init: {t1-t0:.3f}s | frame_bytes: {len(w_single.frame_bytes)/1e6:.0f} MB")
w_single

In [ ]:
# 10 images
t0 = time.perf_counter()
w_ten = Show2D(imgs, title="10× 4096×4096", cmap="inferno")
t1 = time.perf_counter()
print(f"10 images — Init: {t1-t0:.3f}s | frame_bytes: {len(w_ten.frame_bytes)/1e6:.0f} MB")
w_ten

## JS-side timing
Open browser DevTools → Console to see JS timings.
The widget logs `[Show2D] WebGPU FFT initialized` on mount.

Key JS stages for 10×4K:
1. `extractFloat32(frameBytes)` — parse 640 MB DataView → Float32Array
2. Loop: `allFloats.subarray(...)` → 10 copies into `rawDataRef`
3. Loop: `findDataRange()` + `applyColormap()` + `renderToOffscreen()` per image
4. Loop: `drawImage()` per canvas

Stage 3 is the heavy one: for each 4K image, colormapping touches 16M pixels × 4 RGBA bytes = 64 MB per image, 640 MB total RGBA.

## Memory footprint
Where the bytes live:

In [ ]:
n = 10
h = w_ten.height
w_ = w_ten.width
px = h * w_

python_data = n * px * 4  # _data (float32)
python_orig = n * px * 4  # _data_original copies
python_blob = n * px * 4  # frame_bytes (tobytes)
python_total = python_data + python_orig + python_blob

js_float32 = n * px * 4   # rawDataRef (Float32Array)
js_rgba = n * px * 4       # 10 offscreen canvases (ImageData RGBA)
js_total = js_float32 + js_rgba

print(f"Image: {n} × {h}×{w_} float32")
print(f"")
print(f"Python side:")
print(f"  _data:          {python_data/1e6:>7.0f} MB")
print(f"  _data_original: {python_orig/1e6:>7.0f} MB")
print(f"  frame_bytes:    {python_blob/1e6:>7.0f} MB")
print(f"  TOTAL:          {python_total/1e6:>7.0f} MB")
print(f"")
print(f"JS/Browser side:")
print(f"  Float32Array:   {js_float32/1e6:>7.0f} MB  (rawDataRef)")
print(f"  RGBA canvases:  {js_rgba/1e6:>7.0f} MB  (10 offscreen canvases)")
print(f"  TOTAL:          {js_total/1e6:>7.0f} MB")
print(f"")
print(f"Comm transfer:    {python_blob*4/3/1e6:>7.0f} MB  (base64 over websocket)")
print(f"Notebook save:    {python_blob*4/3/1e6:>7.0f} MB  (base64 in .ipynb JSON)")
print(f"")
print(f"Grand total:      {(python_total + js_total)/1e6:>7.0f} MB  (Python + Browser)")